# **Fraud Alert**

A fintech company wants to **flag suspicious transactions**, but fraud is rare.

This creates a fundamental data science challenge known as the class imbalance problem. In a typical transaction dataset, legitimate activities might constitute over 99.9% of the data, while fraudulent ones are a minuscule fraction.

Standard machine learning algorithms, which are designed to maximise overall accuracy, often fail in this context; they can achieve seemingly excellent performance by simply classifying every transaction as legitimate, completely overlooking the rare but critical fraudulent cases they are meant to detect.

The core research problem, therefore, lies in developing a predictive model that is sensitive enough to identify subtle, anomalous fraud patterns without being overwhelmed by the flood of normal data and generating an unmanageable number of false alarms.

### 📊 Dataset Description – Fraud Alert

| **Column Name**         | **Type**            | **Description** |
|--------------------------|---------------------|-----------------|
| **transaction_id**      | Integer             | Unique identifier for each transaction. Used for reference only and not for model training. |
| **amount_usd**          | Numeric             | Transaction amount in US dollars. Larger amounts may increase fraud risk, especially when combined with other suspicious signals. |
| **hour_of_day**         | Integer (0–23)      | Hour when the transaction occurred. Late-night transactions may carry higher fraud risk. |
| **txn_velocity_1h**     | Integer             | Number of transactions made by the account in the last hour. High velocity can indicate suspicious automated activity. |
| **account_age_days**    | Integer             | Number of days since the account was created. Newer accounts are generally higher risk. |
| **failed_logins_7d**    | Integer             | Number of failed login attempts in the past 7 days. Multiple failed attempts may indicate account compromise attempts. |
| **chargebacks_90d**     | Integer             | Number of disputed transactions in the last 90 days. Strong historical indicator of risky behaviour. |
| **country_mismatch**    | Binary (0/1)        | 1 if the transaction country differs from the account’s registered country; otherwise 0. Geographic inconsistencies increase fraud risk. |
| **is_international**    | Binary (0/1)        | 1 if the transaction is international; otherwise 0. International transactions may carry elevated fraud risk. |
| **device_risk_score**   | Numeric (0–100)     | Risk score assigned to the device used for the transaction. Higher scores indicate greater suspicion. |
| **ip_risk_score**       | Numeric (0–100)     | Risk score associated with the IP address used. High-risk IPs increase fraud probability. |
| **merchant_risk_score** | Numeric (0–100)     | Risk score associated with the merchant. Some merchants historically experience higher fraud rates. |
| **fraud**               | Binary (0/1)        | Target variable. 1 = Fraudulent transaction, 0 = Legitimate transaction. |

## Imports

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, make_scorer

## Load the dataset

In [ ]:
df = pd.read_csv("fraud_alert_lite_transactions.csv")
df.head()

,transaction_id,amount_usd,hour_of_day,txn_velocity_1h,account_age_days,failed_logins_7d,chargebacks_90d,country_mismatch,is_international,device_risk_score,ip_risk_score,merchant_risk_score,fraud
0,1,15.89,17,1,305,1,0,0,0,0.9,3.3,14.4,0
1,2,36.32,23,2,2228,1,0,0,0,11.8,12.9,46.6,0
2,3,6.57,15,1,3431,0,0,0,1,61.2,21.5,26.3,0
3,4,91.53,19,0,1262,1,0,0,0,18.7,34.5,20.5,0
4,5,49.93,19,2,1268,0,0,0,0,43.7,8.7,69.7,0


## Quick data check

In [ ]:
df.shape

(3500, 13)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500 entries, 0 to 3499
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   transaction_id       3500 non-null   int64  
 1   amount_usd           3448 non-null   float64
 2   hour_of_day          3500 non-null   int64  
 3   txn_velocity_1h      3500 non-null   int64  
 4   account_age_days     3500 non-null   int64  
 5   failed_logins_7d     3500 non-null   int64  
 6   chargebacks_90d      3500 non-null   int64  
 7   country_mismatch     3500 non-null   int64  
 8   is_international     3500 non-null   int64  
 9   device_risk_score    3448 non-null   float64
 10  ip_risk_score        3448 non-null   float64
 11  merchant_risk_score  3448 non-null   float64
 12  fraud                3500 non-null   int64  
dtypes: float64(4), int64(9)
memory usage: 355.6 KB


In [ ]:
df.isnull().sum()

,0
transaction_id,0
amount_usd,52
hour_of_day,0
txn_velocity_1h,0
account_age_days,0
failed_logins_7d,0
chargebacks_90d,0
country_mismatch,0
is_international,0
device_risk_score,52


There are some missing values for amount_usd, device_risk_score, ip_risk_score, and merchant_risk_score.

All variables are encoded with numeric values, though not necessarily on a numeric scale. Remember that country_mismatch and is_international are binary variables.

In [ ]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
transaction_id,3500.0,1750.500000,1010.507298,1.00,875.750,1750.50,2625.250,3500.00
amount_usd,3448.0,40.608425,37.910027,1.49,16.895,29.90,51.340,431.31
hour_of_day,3500.0,11.746000,6.894526,0.00,6.000,12.00,18.000,23.00
txn_velocity_1h,3500.0,1.217429,1.110569,0.00,0.000,1.00,2.000,6.00
account_age_days,3500.0,1824.534571,1055.754845,2.00,915.250,1810.00,2741.250,3647.00
failed_logins_7d,3500.0,0.381714,0.618850,0.00,0.000,0.00,1.000,3.00
chargebacks_90d,3500.0,0.075714,0.272035,0.00,0.000,0.00,0.000,2.00
country_mismatch,3500.0,0.082000,0.274404,0.00,0.000,0.00,0.000,1.00
is_international,3500.0,0.215714,0.411376,0.00,0.000,0.00,0.000,1.00
device_risk_score,3448.0,25.443097,14.698003,0.00,14.700,25.00,35.300,83.80


Variables are measured using vastly different scales. We'll need to rescale them before using them in our models.

### Class balance

In [ ]:
df["fraud"].unique()

array([0, 1])

In [ ]:
df["fraud"].value_counts()

,count
fraud,
0,3455
1,45


In [ ]:
round((df["fraud"].value_counts(normalize=True) * 100),2)

,proportion
fraud,
0,98.71
1,1.29


Fraud is very rare. Only 1.29% of transactions are fraudulent.

## Preprocessing


In [ ]:
def process_dataset(dataframe):
    """
    Takes a dataframe and does the following:
    - Drops the transaction Id column
    - Fills missing values with 0
    - Splits the data into train and test (75/25)
    - Standardises numerical features
    """
    # Create a copy of the dataframe
    data = dataframe.copy()

    # Drop the transaction Id column
    data = data.drop(columns=["transaction_id"])

    # Fill missing values with 0
    data = data.fillna(0)

    # Separate the data into features (X) and target (y)
    X = data.drop(columns=["fraud"])
    y = data["fraud"]

    # Split the data into train and test (75/25)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

    # Standardise numerical features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
    X_test_scaled = scaler.transform(X_test)
    X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

    return (X_train_scaled, y_train), (X_test_scaled, y_test)

Always scale your data after splitting it into training and testing sets. This prevents data leakage, ensuring the model does not learn information from the test set.

[Reference](https://towardsdatascience.com/data-scaling-101-standardization-and-min-max-scaling-explained-60789833e160/)

In [ ]:
(X_train_scaled, y_train), (X_test_scaled, y_test) = process_dataset(df)

In [ ]:
X_train_scaled.shape, y_train.shape

((2625, 11), (2625,))

In [ ]:
X_train_scaled.head()

,amount_usd,hour_of_day,txn_velocity_1h,account_age_days,failed_logins_7d,chargebacks_90d,country_mismatch,is_international,device_risk_score,ip_risk_score,merchant_risk_score
0,0.946874,-1.554903,1.587732,0.150515,0.970812,-0.276541,-0.297167,-0.515432,2.748323,1.576496,-0.213442
1,-0.565083,0.475610,0.694505,-1.658668,-0.621926,-0.276541,-0.297167,1.940120,-0.880030,-0.977102,-0.483327
2,-0.915016,-0.684683,-0.198722,1.531729,-0.621926,-0.276541,-0.297167,-0.515432,1.525434,-0.819083,-0.425116
3,-0.097886,-0.684683,-0.198722,0.259160,0.970812,-0.276541,-0.297167,-0.515432,0.087531,1.222532,0.008816
4,-0.142722,1.490866,0.694505,-0.599611,-0.621926,-0.276541,-0.297167,-0.515432,-0.382811,1.664987,0.331619


In [ ]:
X_test_scaled.shape, y_test.shape

((875, 11), (875,))

In [ ]:
X_test_scaled.head()

,amount_usd,hour_of_day,txn_velocity_1h,account_age_days,failed_logins_7d,chargebacks_90d,country_mismatch,is_international,device_risk_score,ip_risk_score,merchant_risk_score
0,0.555819,-0.249574,-0.198722,-0.475850,-0.621926,-0.276541,-0.297167,-0.515432,0.483962,-0.894932,-1.615784
1,-0.666959,-0.539647,-0.198722,1.194455,-0.621926,-0.276541,-0.297167,-0.515432,-0.463441,-0.680025,0.172863
2,-0.759815,1.635903,-1.091949,-0.401215,-0.621926,-0.276541,-0.297167,-0.515432,0.195038,0.185923,0.580336
3,-0.648653,-1.554903,-0.198722,1.578966,-0.621926,-0.276541,-0.297167,-0.515432,-0.234989,-0.345023,0.707341
4,-0.716040,1.200793,-1.091949,-1.211805,-0.621926,-0.276541,-0.297167,-0.515432,0.524277,-1.267858,-0.007060


## Baseline model (SVC)

In [ ]:
def train_svc_model(X_train, y_train):
    """
    This function takes X_train and y_train data and trains an SVC model.
    """
    model = SVC(random_state=40).fit(X_train, y_train)
    return model

In [ ]:
svc_baseline = train_svc_model(X_train_scaled, y_train)

In [ ]:
svc_baseline

SVC(random_state=40)

### Evaluate baseline model on the test set

In [ ]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    TP = cm[0][1]
    FN = cm[1][0]
    TN = cm[0][0]
    FP = cm[1,1]
    cr = classification_report(y_test, y_pred)
    roc = roc_auc_score(y_test, y_pred)
    return print(f'TP: {TP}'), print(f'FN: {FN}'), print(f'TN: {TN}'), print(f'FP: {FP}'), print(f'Classification Report:\n {cr}'), print(f'ROC-AUC Score: {roc}')

In [ ]:
evaluate_model(svc_baseline, X_test_scaled, y_test)

TP: 0
FN: 13
TN: 862
FP: 0
Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99       862
           1       0.00      0.00      0.00        13

    accuracy                           0.99       875
   macro avg       0.49      0.50      0.50       875
weighted avg       0.97      0.99      0.98       875

ROC-AUC Score: 0.5


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


(None, None, None, None, None, None)

From the ROC-AUC score of 0.5, our model is no better than random guessing (e.g., flipping a coin).

## Custom scoring: Log Loss
Log loss penalises confident wrong predictions.

We clip predictions to avoid log(0).  

In GridSearchCV, we set `greater_is_better=False` because lower log loss is better.


In [ ]:
def custom_log_loss(y_true, y_pred):
    eps = 1e-15
    y_pred = np.clip(y_pred, eps, 1 - eps)
    loss = -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))
    return float(np.round(loss, 7))

In [ ]:
y_pred = svc_baseline.predict(X_test_scaled)

In [ ]:
custom_log_loss(y_test, y_pred)

0.5131475

## Hyperparameter tuning with GridSearchCV (SVC)


In [ ]:
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1, 10, 100],
    'class_weight': [None, 'balanced'],
}

scorer = make_scorer(custom_log_loss, greater_is_better=False)

grid = GridSearchCV(
    estimator=SVC(),
    param_grid=param_grid,
    scoring=scorer,
    cv=5
)

In [ ]:
grid.fit(X_train_scaled, y_train)

GridSearchCV(cv=5, estimator=SVC(),
             param_grid={'C': [0.001, 0.01, 0.1, 1, 10, 100],
                         'class_weight': [None, 'balanced'],
                         'gamma': [0.001, 0.01, 0.1, 1, 10, 100]},
             scoring=make_scorer(custom_log_loss, greater_is_better=False, response_method='predict'))

In [ ]:
grid.best_estimator_

SVC(C=0.001, gamma=0.001)

In [ ]:
best_svc = grid.best_estimator_

### Evaluate tuned model

In [ ]:
print('best_svc:')
evaluate_model(best_svc, X_test_scaled, y_test)
y_pred = best_svc.predict(X_test_scaled)
print("Custom Log Loss:", custom_log_loss(y_test, y_pred))

print('svc_baseline:')
evaluate_model(svc_baseline, X_test_scaled, y_test)
y_pred = svc_baseline.predict(X_test_scaled)
print("Custom Log Loss:", custom_log_loss(y_test, y_pred))

best_svc:
TP: 0
FN: 13
TN: 862
FP: 0
Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99       862
           1       0.00      0.00      0.00        13

    accuracy                           0.99       875
   macro avg       0.49      0.50      0.50       875
weighted avg       0.97      0.99      0.98       875

ROC-AUC Score: 0.5
Custom Log Loss: 0.5131475
svc_baseline:
TP: 0
FN: 13
TN: 862
FP: 0
Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99       862
           1       0.00      0.00      0.00        13

    accuracy                           0.99       875
   macro avg       0.49      0.50      0.50       875
weighted avg       0.97      0.99      0.98       875

ROC-AUC Score: 0.5
Custom Log Loss: 0.5131475


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/m

The tuned model performs no better than the baseline model.